In [1]:
# Import packages

import pandas as pd
import numpy as np
import lightgbm as lgb
from datetime import datetime
import os

from rdkit import Chem 
from rdkit.Chem import AllChem, MACCSkeys 
from sklearn.preprocessing import MinMaxScaler

In [2]:
# Input smiles to predict

# smiles_file = './test_smiles_file.txt'
data_warrior_3D_file = './test_final_model.sdf'

suppl = Chem.SDMolSupplier(data_warrior_3D_file)

SMILES = []
for mol in suppl:
    if mol is None:   # skip malformed molecules
        continue
    smiles = Chem.MolToSmiles(mol)
    SMILES.append(smiles)

[20:52:52] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:52:52] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:52:52] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:52:52] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:52:52] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


In [3]:
# Add all features

# Add ECFP Descriptors
fpgen = AllChem.GetMorganGenerator(radius=2)
ecfp_fingerprints = []
for smile in SMILES:
    mol = Chem.MolFromSmiles(smile)
    fp = Chem.rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, 2).ToBitString()
    # print(list(fp))
    ecfp_fingerprints.append(list(fp))

ecfp_col_names = [f"ECFP_{i}" for i in range(0,2048)]
df = pd.DataFrame(ecfp_fingerprints, columns=ecfp_col_names)


# Add Mordred Descriptors
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
unique_mordred_filename = f'mordred_labelled_dataset_{timestamp}.csv'

cmd = f'python -m mordred -3 {data_warrior_3D_file} -o {unique_mordred_filename}'
os.system(cmd)

mordred_df = pd.read_csv(unique_mordred_filename)

# Add MACCS Descriptors
def get_maccs_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return list(MACCSkeys.GenMACCSKeys(mol).ToList())

col_names = [f"MACCS_{i}" for i in range(0,167)]
maccs_fps = []
for smile in SMILES: 
    maccs_fps.append(get_maccs_fingerprint(smile))

maccs_fp_df = pd.DataFrame(data=maccs_fps, columns=col_names)

all_labelled_df = pd.concat([df, mordred_df, maccs_fp_df], axis=1)

cols_in_og = list(pd.read_csv(f'./bbbx_training_set.csv').columns)
cols_to_keep = [col for col in list(all_labelled_df.columns) if col in cols_in_og]

fully_labelled_df = all_labelled_df.loc[:, cols_to_keep]
fully_labelled_df.to_csv(f'./fully_labelled_dataset_{timestamp}.csv')
print(f'The fully labelled input dataset was saved to: ./fully_labelled_dataset_{timestamp}.csv', )
X_all = fully_labelled_df.values



[20:52:52] DEPRECATION WARNING: please use MorganGenerator
[20:52:52] DEPRECATION WARNING: please use MorganGenerator
[20:52:52] DEPRECATION WARNING: please use MorganGenerator
[20:52:52] DEPRECATION WARNING: please use MorganGenerator
[20:52:52] DEPRECATION WARNING: please use MorganGenerator
[20:52:52] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:52:52] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:52:52] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:52:52] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[20:52:52] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[ERROR] Compound 1: module 'numpy' has no attribute 'float'.
`np.float` was a deprecated alias for the builtin `float`. To avoid this error in ex

The fully labelled input dataset was saved to: ./fully_labelled_dataset_20260806_205252.csv


In [ ]:
X_all = fully_labelled_df.values

all_predictions = []
for model_id in range(0,5): 
    # TODO: Add the 15 values here
    model = lgb.Booster(model_file=f'./models/custom_mtl_original_outer_{model_id}.txt')
    predictions = model.predict(X_all)
    all_predictions.append(predictions)

all_preds = np.mean(np.array(all_predictions), axis=0)

scaler = MinMaxScaler()
scaler.fit(np.array([-2.7, 1, 1.7]).reshape(-1, 1))
y_pred_norm = scaler.transform(all_preds.reshape(-1, 1)).flatten()
final_scores = y_pred_norm*6

print(SMILES)

output_df_data = {'smiles': SMILES, 'predicted_logBB': all_preds, 'BBBX_score': final_scores}
output_df = pd.DataFrame(data=output_df_data)
unique_file_name = f'./predicted_scores_{timestamp}.csv'
output_df.to_csv(unique_file_name)

print(f'Predictions were saved to {unique_file_name}')

[LightGBM] [Fatal] The number of features in data (3041) is not the same as it was in training data (2926).
You can set ``predict_disable_shape_check=true`` to discard this error, but please be aware what you are doing.


LightGBMError: The number of features in data (3041) is not the same as it was in training data (2926).
You can set ``predict_disable_shape_check=true`` to discard this error, but please be aware what you are doing.